# Lab 05-02 — Multi-query: one question in, several search queries out

**Track 05 · Query transformation** — a single user question is a single point in embedding space, but the answer may live near several DIFFERENT points: the same fact phrased as "capital of Uruguay", "Montevideo", "Uruguay's largest city" … each lands in a slightly different region of the vector store. Top-k retrieval samples ONE region, so phrasing luck decides what comes back.

This notebook is **self-contained**: it imports LangChain, sentence-transformers, and faiss directly — no repo component library. Every block of the pipeline is built right here:

```
user question ──► ChatGroq (query generator) ──► 3+ search-query variants
                        │                              │
                        ▼                              ▼
        base_retriever (FAISS top-3) ◄── each variant retrieved separately
                        │
                        ▼
        MultiQueryRetriever merges the per-variant results into one
        deduplicated union (include_original=True: never worse than top-k)
```

Multi-query expansion is LangChain's `MultiQueryRetriever` from `langchain-classic` — the same class the lab uses, so nothing here is re-implemented: the LLM rewrites the user question into 3+ search-query variants, every variant is retrieved, and the results are merged into one deduplicated union. A fact reachable through ANY phrasing now has a chance to surface.

The retrieval machinery is LangChain-native this time: `MultiQueryRetriever` wraps a LangChain `BaseRetriever`, so the inner retriever is `store.as_retriever(search_kwargs={"k": TOP_K})` over the same local BGE embeddings. Same three questions as lab 01 (1606/1610/1626) so you can compare the transformations directly.


## Setup

One prerequisite must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/passages.parquet` + `test.parquet`, already fetched by the repo's manifest-verified fetchers.
- **`GROQ_API_KEY` in the repo-root `.env`** — the Groq LLM is the query *generator* (it never embeds); embeddings stay local BGE.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-huggingface`, `langchain-community`, `langchain-classic`, `langchain-groq`, `sentence-transformers`, and `faiss-cpu`. The bootstrap cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
%pip install -q sentence-transformers langchain-huggingface langchain-community langchain-classic faiss-cpu langchain-groq python-dotenv pandas


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# LangChain + sentence-transformers + faiss — the only libraries this
# notebook needs. Nothing is imported from the repo's src/ component library.
from langchain_classic.retrievers.multi_query import (  # noqa: E402
    MultiQueryRetriever,
)
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_groq import ChatGroq  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402
# (Gemini alternative: from langchain_google_genai import ChatGoogleGenerativeAI)

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)

load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives in the repo-root .env


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_PASSAGES = 100` takes a deterministic head of the 3200-passage corpus; `QUESTION_IDS = [1606, 1610, 1626]` reuses lab 01's questions so the transformations are directly comparable; `TOP_K = 3` is the per-variant retrieval depth (the union is bigger than k); `LLM_MODEL` names the Groq model that generates the variants (never the embedder); `BGE_MODEL_NAME` pins the local embedder. `PREVIEW` truncates the passage previews the demo prints next to each hit.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus (keeps runtime low)
QUESTION_IDS = [1606, 1610, 1626]  # same questions as lab 01, for comparison
TOP_K = 3  # per-variant retrieval depth; the union is bigger than k
LLM_MODEL = "llama-3.3-70b-versatile"  # Groq is the query *generator*, never the embedder
# (Gemini alternative: LLM_MODEL = "gemini-2.5-flash" — needs GOOGLE_API_KEY in .env)
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
PREVIEW = 62  # max characters of passage text shown next to each hit


## 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files

`load_passages` reads the first `n` passages (text + ids) from `passages.parquet`; `load_questions` pulls specific rows by id from `test.parquet`; `preview` flattens a passage onto one line for printing. Identical helpers to labs 01/04/05/06 keep the track's experiments directly comparable — the only thing that changes is the retriever wrapper.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


## 3. Experiment — LLM query variants -> per-variant retrieval -> union

`run_experiment` embeds the 100-passage subset once (batched BGE) and builds the FAISS store natively (`FAISS.from_documents` — embed + index in one call, separately timed from the outside), then wires the two LangChain objects: the base retriever is `store.as_retriever(search_kwargs={"k": TOP_K})`, and `MultiQueryRetriever.from_llm(retriever=..., llm=ChatGroq, include_original=True)` sits on top — `include_original=True` means the user's own query joins the LLM's variants, so the union can never be WORSE than plain top-k. For each question we capture the generated variants (one `llm_chain` call, timed) and the merged, deduplicated union (the retriever call, timed).

The LLM section reports **run/skip** explicitly: with `GROQ_API_KEY` in `.env` the generator runs (`ChatGroq`, one call per question); without the key it prints SKIP and the multi-query machinery cannot generate variants — the same contract the lab's `GroqLLM` follows when the key is missing.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — LLM query variants -> per-variant retrieval -> union
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)

    # --- Embed locally (BGE) and index in-memory with langchain-native FAISS -
    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME, encode_kwargs={"normalize_embeddings": True}
    )
    t0 = time.perf_counter()
    store = FAISS.from_documents(chunks, embedder)
    index_s = time.perf_counter() - t0

    # --- Base retriever (LangChain contract) + the multi-query wrapper --------
    base_retriever = store.as_retriever(search_kwargs={"k": TOP_K})
    # include_original=True: the user's own query joins the LLM's variants, so
    # the union can never be WORSE than plain top-k.
    if os.getenv("GROQ_API_KEY"):
        llm = ChatGroq(model=LLM_MODEL, temperature=0.0)
        llm_status = f"run (ChatGroq {LLM_MODEL})"
    else:
        llm = None
        llm_status = ("skip (no GROQ_API_KEY in the repo-root .env — "
                      "query variants cannot be generated)")
        print("LLM section: SKIP —", llm_status)
    multi_retriever = MultiQueryRetriever.from_llm(
        retriever=base_retriever, llm=llm, include_original=True
    )

    # --- Per question: generated variants + the merged union ------------------
    results = []
    for qid, qtext in questions:
        t0 = time.perf_counter()
        variants = multi_retriever.llm_chain.invoke({"question": qtext})
        gen_s = time.perf_counter() - t0
        t0 = time.perf_counter()
        union = multi_retriever.invoke(qtext)
        union_s = time.perf_counter() - t0
        results.append(
            {
                "qid": qid,
                "question": qtext,
                "variants": variants,
                "gen_s": gen_s,
                "union": union,
                "union_s": union_s,
            }
        )

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "indexed": len(passage_texts),
        "index_s": index_s,
        "llm_status": llm_status,
        "results": results,
    }


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact from three angles: the corpus subset with the index timing and the LLM section's run/skip status; per question, the generated variants with their generation time, then the merged union — how many unique docs came back, how long the merge took, the top-3 with passage ids and previews, and a count of the docs beyond top-3; then a takeaway on the trade: one cheap LLM call for several retrieval passes over different phrasings, merged, with the union never worse than plain top-k.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 02 — Multi-query: one question in, several search queries out")
    print(f"{BGE_MODEL_NAME} (local) -> FAISS top-{TOP_K} -> {LLM_MODEL} generator")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {exp['indexed']} passages (first {N_PASSAGES} of 3200, ids {exp['passage_ids'][0]}..{exp['passage_ids'][-1]})")
    print(f"    FAISS index built in {exp['index_s']:.3f}s over local BGE embeddings")
    print(f"    LLM section: {exp['llm_status']}")

    print(f"\n[2] Generated variants -> union (per question):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"]}] "{r["question"]}"')
        print(f"      variants ({len(r['variants'])}, {r['gen_s']:.1f}s):")
        for v in r["variants"]:
            print(f"        - {v}")
        print(f"      union: {len(r['union'])} unique docs "
              f"(k={TOP_K} per variant, {r['union_s']:.1f}s)")
        for rank, doc in enumerate(r["union"][:TOP_K], 1):
            pid = doc.metadata.get("id", "?")
            print(f"        {rank}. [passage {pid}] {preview(doc.page_content)}")
        if len(r["union"]) > TOP_K:
            print(f"        … {len(r['union']) - TOP_K} more unique docs beyond top-{TOP_K}")

    print("\n[3] Takeaway")
    print("    Multi-query trades one cheap LLM call per question for several")
    print("    retrieval passes over different phrasings, then merges the")
    print("    unique results. The union is never worse than plain top-k")
    print("    (the original query is included), and a fact reachable only")
    print("    through a different phrasing finally has a chance to surface.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: exactly `N_PASSAGES` passages indexed; per question — at least one generated variant, a deduplicated union, a union size >= `TOP_K` (guaranteed by `include_original=True`), and the content checks that the union still carries the answer's keyword (montevideo / spanish / 1930). Multi-query properties are pinned to what survives LLM wording variance: counts and dedup, not exact phrasings. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Structural properties (no LLM involved).
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))

    # Multi-query properties, pinned to what survives LLM wording variance.
    for r in exp["results"]:
        tag = f"Q{r['qid']}"

        # The generator must return at least one variant.
        checks.append((f"{tag} generated >=1 query variant", len(r["variants"]) >= 1))

        # The union is deduplicated: unique page content, no repeats.
        contents = [d.page_content for d in r["union"]]
        checks.append((f"{tag} union is deduplicated",
                       len(contents) == len(set(contents))))

        # include_original=True guarantees the union covers plain top-k, so it
        # is always >= TOP_K distinct documents.
        checks.append((f"{tag} union size >= TOP_K", len(r["union"]) >= TOP_K))

        # Content check: the union must carry the answer's keyword.
        joined = " ".join(d.page_content for d in r["union"]).lower()
        if r["qid"] == 1606:
            kw = "montevideo"
        elif r["qid"] == 1610:
            kw = "spanish"
        else:  # 1626
            kw = "1930"
        checks.append((f"{tag} union retains '{kw}'", kw in joined))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A minute of embedding + 3 Groq variant-generation calls on the 100-passage subset — no downloads. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The three questions with the LLM's search-query variants, and the merged union the retriever actually returned — how many unique passages surfaced through different phrasings, and what sits at the top.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the parquet files are intact and the LLM section ran (see the status line in the demo).


In [ ]:
verify_gate(exp)
